In [5]:
import argparse
from datetime import datetime, timezone
import pandas as pd

from aind_data_schema.components.identifiers import Code, DataAsset
from aind_data_schema.core.processing import (
    DataProcess,
    Processing,
    ProcessName,
    ProcessStage,
    ResourceTimestamped,
    ResourceUsage,
)
from aind_data_schema_models.units import MemoryUnit
from aind_data_schema_models.system_architecture import OperatingSystem, CPUArchitecture
from codeocean import CodeOcean
import os
# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
import os
import sys
_anchor = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.path.abspath(os.getcwd())
while _anchor != os.path.dirname(_anchor):
    _beh_ephys_root = os.path.join(_anchor, "code", "beh_ephys_analysis")
    if os.path.isdir(os.path.join(_beh_ephys_root, "utils")):
        if _beh_ephys_root in sys.path:
            sys.path.remove(_beh_ephys_root)
        sys.path.insert(0, _beh_ephys_root)
        break
    _anchor = os.path.dirname(_anchor)
from utils.beh_functions import parseSessionID
import json
from utils.capsule_migration import CAPSULE_ROOT

# Generating fiber location data's procesing metadata

In [6]:
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))

In [ ]:
computation_params_file = CAPSULE_ROOT + '/code/data_management/processing_params.json'
with open(computation_params_file, 'r') as f:
    computation_params = json.load(f)

## CCF for fibers

In [ ]:
ccf_data_asset_id = '66668438-3b32-4160-b571-037e117d8568'
ts = client.data_assets.get_data_asset(data_asset_id=ccf_data_asset_id).created
t = datetime.fromtimestamp(ts, tz=timezone.utc)

In [ ]:
data_asset_ids = ['']

data_asset_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name  for id in data_asset_ids]
data_assets = [DataAsset(url=name) for name in data_asset_names_full]

In [ ]:
dp_list = []
# generate location files for each animal
curr_code = Code(
    url='https://github.com/AllenNeuralDynamics/ccf_annotations_data_prep',
    run_script='code/photometry.ipynb',
    version='da10a6e895b00f7722b834a092c14b95f434ecc7',
    input_data=data_assets,
)
curr_dp = DataProcess(
            process_type=ProcessName.OTHER,
            name='Convert manually labeling to ccf locations',
            experimenters=["Sue Su"],
            stage=ProcessStage.ANALYSIS,
            start_date_time=t,
            output_path='',
            code=curr_code,
            notes='Convert manually labeling to ccf locations',
            )
dp_list.append(curr_dp)
# generate combined csv file
curr_code = Code(
    url='https://github.com/AllenNeuralDynamics/ccf_annotations_data_prep',
    run_script='code/combine_photometry.ipynb',
    version='da10a6e895b00f7722b834a092c14b95f434ecc7',
    input_data='',
)
curr_dp = DataProcess(
            process_type=ProcessName.OTHER,
            name="Combine different animals' fiber location",
            experimenters=["Sue Su"],
            stage=ProcessStage.ANALYSIS,
            start_date_time=t,
            output_path='',
            code=curr_code,
            notes="Combine different animals' fiber location",
            )
dp_list.append(curr_dp) 

p_ccf_meta = Processing.create_with_sequential_process_graph(
    data_processes=dp_list)